# 03 · Exploración manual de variables
Cada hipótesis se evalúa con ΔAUC pareado (mismos folds) frente a la base, con XGBoost fijo.
Regla: si el ΔAUC no supera ~2 errores estándar, se descarta.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from features.features import ID, TARGET, prepare  # noqa: E402

train = pd.read_csv(ROOT / "data/raw/train.csv")
test = pd.read_csv(ROOT / "data/raw/test.csv")
y = train[TARGET].astype(int)
RAW = [c for c in train.columns if c not in (TARGET, ID)]
X_fe = prepare(train)

XGB = dict(n_estimators=300, learning_rate=0.05, max_depth=4, min_child_weight=5,
           subsample=0.8, colsample_bytree=0.8, tree_method="hist", eval_metric="auc",
           random_state=42, n_jobs=-1)
SPLITS = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42).split(X_fe, y))
xgb_model = lambda: xgb.XGBClassifier(**XGB)


def fold_aucs(X, factory=xgb_model):
    return np.array([
        roc_auc_score(y.iloc[va], factory().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[va])[:, 1])
        for tr, va in SPLITS
    ])


BASE = fold_aucs(X_fe)
results = [dict(variante="base (features actuales)", auc=BASE.mean(), delta=0.0, se=0.0, folds_mejores=None)]


def record(name, a):
    d = a - BASE
    results.append(dict(variante=name, auc=a.mean(), delta=d.mean(),
                        se=d.std(ddof=1) / np.sqrt(len(d)), folds_mejores=int((d > 0).sum())))
    return pd.DataFrame(results).round(5)


def compare(name, X_new):
    return record(name, fold_aucs(X_new))


print("AUC base (10 folds):", BASE.mean().round(5), "±", BASE.std().round(5))

AUC base (10 folds): 0.86519 ± 0.00545


## 1. Mirada a mano: picos, valores exactos y sanidad del Id

In [2]:
u, inc = train["RevolvingUtilizationOfUnsecuredLines"], train["MonthlyIncome"]
masks = {"util == 0": u == 0, "0 < util < 0.05": (u > 0) & (u < 0.05), "util == 1": u == 1,
         "0.95 <= util < 1": (u >= 0.95) & (u < 1), "util > 1": u > 1}
for name, m in masks.items():
    print(f"{name:18s} n={m.sum():6d} ({m.mean():.1%})  tasa de impago={y[m].mean():.3f}")
r1000 = (inc > 0) & (inc % 1000 == 0)
r500 = (inc > 0) & (inc % 500 == 0) & ~r1000
rest = inc.notna() & (inc > 0) & ~r1000 & ~r500
for name, m in {"ingreso múltiplo de 1000": r1000, "múltiplo de 500": r500, "otro": rest}.items():
    print(f"{name:26s} n={m.sum():6d}  tasa de impago={y[m].mean():.3f}")
print("AUC de Id como predictor (esperado ~0.5):", round(roc_auc_score(y, train[ID]), 4))

util == 0          n=  7652 (7.3%)  tasa de impago=0.030
0 < util < 0.05    n= 26038 (24.8%)  tasa de impago=0.014
util == 1          n=    14 (0.0%)  tasa de impago=0.214
0.95 <= util < 1   n=  9718 (9.3%)  tasa de impago=0.196
util > 1           n=  2316 (2.2%)  tasa de impago=0.367
ingreso múltiplo de 1000   n= 12624  tasa de impago=0.061
múltiplo de 500            n=  7557  tasa de impago=0.062
otro                       n= 62848  tasa de impago=0.073
AUC de Id como predictor (esperado ~0.5): 0.5023


## 2. Familias de variables candidatas (sin usar el objetivo)

In [3]:
age_bin = pd.qcut(X_fe["age"], 10, labels=False, duplicates="drop")


def rel(c):
    return X_fe[c] - X_fe.groupby(age_bin)[c].transform("median")


cohort = pd.DataFrame({f"{c}_rel_age": rel(c) for c in
                       ["log_income", "util_clip", "NumberOfOpenCreditLinesAndLoans", "total_late"]})
cohort["income_pct_age"] = X_fe.groupby(age_bin)["log_income"].rank(pct=True)
cohort["util_pct_age"] = X_fe.groupby(age_bin)["util_clip"].rank(pct=True)

spikes = pd.DataFrame({
    "util_eq0": (u == 0).astype(int), "util_eq1": (u == 1).astype(int),
    "util_095_1": ((u >= 0.95) & (u <= 1)).astype(int),
    "inc_round1000": r1000.astype(int), "inc_round500": r500.astype(int)})

n_open = X_fe["NumberOfOpenCreditLinesAndLoans"]
late90 = X_fe["NumberOfTimes90DaysLate"].fillna(0)
late60 = X_fe["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0)
exposure = pd.DataFrame({
    "rev_exposure": X_fe["util_clip"] * n_open,
    "debt_per_line": X_fe["debt_amount"] / (1 + n_open),
    "income_per_line": X_fe["MonthlyIncome"] / (1 + n_open),
    "re_share": X_fe["NumberRealEstateLoansOrLines"] / (1 + n_open),
    "has_90": (late90 > 0).astype(int),
    "only_mild_late": ((X_fe["total_late"] > 0) & (late90 == 0) & (late60 == 0)).astype(int),
    "late90_share": X_fe["NumberOfTimes90DaysLate"] / X_fe["total_late"].where(X_fe["total_late"] > 0)})

miss = pd.DataFrame({"miss_pattern": X_fe["income_missing"] * 2 + X_fe["dependents_missing"]})
print(pd.crosstab(X_fe["income_missing"], X_fe["dependents_missing"]))

cnt = pd.concat([train["MonthlyIncome"], test["MonthlyIncome"]]).value_counts()
freq = pd.DataFrame({"income_freq": inc.map(cnt).fillna(0)})

dependents_missing      0     1
income_missing                 
0                   84172     0
1                   18057  2771


In [4]:
families = {"+ cohorte por edad": cohort, "+ picos y redondez": spikes,
            "+ exposición y composición": exposure, "+ patrón de faltantes": miss,
            "+ frecuencia de ingreso (train+test)": freq}
for name, extra in families.items():
    out = compare(name, pd.concat([X_fe, extra], axis=1))
out

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0


## 3. Imputación: NaN nativo de XGBoost vs. mediana global

In [5]:
compare("imputación: mediana global", X_fe.fillna(X_fe.median()))

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0
6,imputación: mediana global,0.86521,0.00002,0.00011,5.0


## 4. Riesgo por segmento (análogo al ranking de lluvia), calculado dentro de cada fold
Tasa de impago suavizada por celda (edad × utilización, atrasos × utilización).
En entrenamiento se codifica con OOF interno; en validación, con todo el train del fold.

In [6]:
SEGS = {
    "seg_age_util": (("age", [17, 30, 40, 50, 60, 70, 110]), ("util_clip", [-0.01, 0.1, 0.3, 0.6, 1, 2])),
    "seg_late_util": (("total_late", [-1, 0, 1, 2, 100]), ("util_clip", [-0.01, 0.1, 0.3, 0.6, 1, 2])),
}


def _key(X, spec):
    (c1, b1), (c2, b2) = spec
    a = pd.cut(X[c1], b1, labels=False).fillna(-1).astype(int)
    b = pd.cut(X[c2], b2, labels=False).fillna(-1).astype(int)
    return a * 100 + b


def _enc(key_tr, y_tr, key_apply, alpha):
    prior = y_tr.mean()
    g = pd.DataFrame({"k": key_tr.values, "y": y_tr.values}).groupby("k")["y"].agg(["sum", "count"])
    rate = (g["sum"] + alpha * prior) / (g["count"] + alpha)
    return key_apply.map(rate).fillna(prior).values


def add_segment_risk(X_tr, y_tr, X_va, alpha=50, inner=5, seed=0):
    tr, va = X_tr.copy(), X_va.copy()
    skf = StratifiedKFold(inner, shuffle=True, random_state=seed)
    for name, spec in SEGS.items():
        k_tr, k_va = _key(X_tr, spec), _key(X_va, spec)
        oof = np.zeros(len(X_tr))
        for a, b in skf.split(X_tr, y_tr):
            oof[b] = _enc(k_tr.iloc[a], y_tr.iloc[a], k_tr.iloc[b], alpha)
        tr[name] = oof
        va[name] = _enc(k_tr, y_tr, k_va, alpha)
    return tr, va


seg = []
for tr, va in SPLITS:
    Xtr, Xva = add_segment_risk(X_fe.iloc[tr], y.iloc[tr], X_fe.iloc[va])
    m = xgb.XGBClassifier(**XGB).fit(Xtr, y.iloc[tr])
    seg.append(roc_auc_score(y.iloc[va], m.predict_proba(Xva)[:, 1]))
record("+ riesgo por segmento (en el fold)", np.array(seg))

,variante,auc,delta,se,folds_mejores
0,base (features actuales),0.86519,0.00000,0.00000,NaN
1,+ cohorte por edad,0.86498,-0.00021,0.00016,3.0
2,+ picos y redondez,0.86520,0.00001,0.00010,5.0
3,+ exposición y composición,0.86491,-0.00028,0.00018,5.0
4,+ patrón de faltantes,0.86507,-0.00013,0.00013,4.0
5,+ frecuencia de ingreso (train+test),0.86512,-0.00007,0.00009,4.0
6,imputación: mediana global,0.86521,0.00002,0.00011,5.0
7,+ riesgo por segmento (en el fold),0.86513,-0.00006,0.00017,5.0


## 5. Valor de la limpieza para un modelo lineal

In [7]:
logreg = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                               LogisticRegression(C=1.0, max_iter=3000))
lin_cols = ["util_clip", "log_income", "log_debt_amount", "debt_ratio_valid", "age", "total_late",
            "max_late", "any_late", "late_severity", "late_sentinel", "income_missing", "income_zero",
            "dependents_missing", "non_re_lines", "NumberOfDependents",
            "NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines", "util_gt1"]
X_lin = X_fe[lin_cols].copy()
X_lin[["total_late", "max_late", "late_severity"]] = X_lin[["total_late", "max_late", "late_severity"]].clip(upper=10)
lin = pd.DataFrame({"modelo": ["logreg · crudas", "logreg · limpias y transformadas"],
                    "auc": [fold_aucs(train[RAW], logreg).mean(), fold_aucs(X_lin, logreg).mean()]})
lin.round(5)

,modelo,auc
0,logreg · crudas,0.69961
1,logreg · limpias y transformadas,0.85640


## 6. Ablación hacia atrás: ¿cuál es el set mínimo que no pierde AUC?

In [ ]:
from features.features import clean  # noqa: E402

clean_df = clean(train.drop(columns=[TARGET, ID]))
GROUPS = {
    "agregados de atrasos": ["total_late", "max_late", "any_late", "late_severity"],
    "derivados de ingreso y deuda": ["log_income", "debt_ratio_valid", "debt_amount",
                                     "log_debt_amount", "income_per_dep"],
    "utilización derivada": ["util_clip", "util_gt1"],
    "interacciones y estructura": ["non_re_lines", "age_x_util", "late_x_util"],
}
compare("solo crudas (sin limpieza)", train[RAW])
compare("crudas limpias + banderas", clean_df)
compare("mínimo: limpias + total_late + any_late",
        pd.concat([clean_df, X_fe[["total_late", "any_late"]]], axis=1))
for g, cols in GROUPS.items():
    out = compare(f"base sin {g}", X_fe.drop(columns=cols))
out

## 7. Regresión logística: qué features a mano ayudan a un modelo lineal

In [ ]:
extra_lin = pd.concat([X_lin, spikes[["util_eq0", "util_095_1"]],
                       exposure[["has_90", "only_mild_late"]]], axis=1)
lin2 = pd.DataFrame({
    "modelo": ["logreg · limpias y transformadas", "+ util_eq0, util_095_1, has_90, only_mild_late"],
    "auc": [fold_aucs(X_lin, logreg).mean(), fold_aucs(extra_lin, logreg).mean()]})
lin2.round(5)

## 8. Decisiones (completar)
- Familias descartadas (ΔAUC dentro de ±2 SE): cohorte por edad, picos y redondez, exposición y composición,
  patrón de faltantes, frecuencia de ingreso, riesgo por segmento en el fold.
- Imputación: NaN nativo (la mediana no cambia el AUC).
- Set mínimo candidato:
- Qué limpieza solo importa para modelos lineales: colas, centinelas y transformaciones (0.700 a 0.856).
## 6. Ablación hacia atrás: ¿cuál es el set mínimo que no pierde AUC?

In [ ]:
from features.features import clean  # noqa: E402

clean_df = clean(train.drop(columns=[TARGET, ID]))
GROUPS = {
    "agregados de atrasos": ["total_late", "max_late", "any_late", "late_severity"],
    "derivados de ingreso y deuda": ["log_income", "debt_ratio_valid", "debt_amount",
                                     "log_debt_amount", "income_per_dep"],
    "utilización derivada": ["util_clip", "util_gt1"],
    "interacciones y estructura": ["non_re_lines", "age_x_util", "late_x_util"],
}
compare("solo crudas (sin limpieza)", train[RAW])
compare("crudas limpias + banderas", clean_df)
compare("mínimo: limpias + total_late + any_late",
        pd.concat([clean_df, X_fe[["total_late", "any_late"]]], axis=1))
for g, cols in GROUPS.items():
    out = compare(f"base sin {g}", X_fe.drop(columns=cols))
out

## 7. Regresión logística: qué features a mano ayudan a un modelo lineal

In [ ]:
extra_lin = pd.concat([X_lin, spikes[["util_eq0", "util_095_1"]],
                       exposure[["has_90", "only_mild_late"]]], axis=1)
lin2 = pd.DataFrame({
    "modelo": ["logreg · limpias y transformadas", "+ util_eq0, util_095_1, has_90, only_mild_late"],
    "auc": [fold_aucs(X_lin, logreg).mean(), fold_aucs(extra_lin, logreg).mean()]})
lin2.round(5)

## 8. Decisiones (completar)
- Familias descartadas (ΔAUC dentro de ±2 SE): cohorte por edad, picos y redondez, exposición y composición,
  patrón de faltantes, frecuencia de ingreso, riesgo por segmento en el fold.
- Imputación: NaN nativo (la mediana no cambia el AUC).
- Set mínimo candidato:
- Qué limpieza solo importa para modelos lineales: colas, centinelas y transformaciones (0.700 a 0.856).